<a href="https://colab.research.google.com/github/CienciaDatosUdea/002_EstudiantesAprendizajeEstadistico/blob/main/semestre2026-1/Laboratorios/Laboratorio_09_nn_zeroV1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9: Red Neuronal desde Cero (From Scratch)

En este laboratorio construiremos una red neuronal multicapa completamente desde cero usando NumPy.  
El objetivo es clasificar imágenes de **gatos vs. no-gatos** implementando:

1. Clase `layer_nn` con inicialización de parámetros
2. Funciones de activación (ReLU, Sigmoid, Tanh) y sus derivadas
3. **Forward Propagation** generalizado para L capas
4. **Función de Costo** (Binary Cross-Entropy)
5. **Backward Propagation** generalizado para L capas
6. **Gradient Descent** para actualizar parámetros
7. Entrenamiento y evaluación del modelo

---
## 0. Importaciones

In [ ]:
import pandas as pd
import numpy as np
import h5py
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Librerías cargadas correctamente ✓")

---
## 1. Carga y Exploración del Dataset

El dataset contiene imágenes RGB de 64×64 píxeles etiquetadas como:
- **1** → gato
- **0** → no gato

In [ ]:
data_train = "train_catvnoncat.h5"
train_dataset = h5py.File(data_train, "r")

data_test = "test_catvnoncat.h5"
test_dataset = h5py.File(data_test, "r")

print("Keys train:", list(train_dataset.keys()))
print("Keys test: ", list(test_dataset.keys()))

In [ ]:
# Leer los datos
xtrain_classes, xtrain, train_label = (
    train_dataset["list_classes"],
    train_dataset["train_set_x"],
    train_dataset["train_set_y"]
)

test_classes, xtest, test_label = (
    test_dataset["list_classes"],
    test_dataset["test_set_x"],
    test_dataset["test_set_y"]
)

print("Forma del conjunto de entrenamiento (X):", np.shape(xtrain))
print("Forma del conjunto de test (X):         ", np.shape(xtest))
print("Forma de las etiquetas de entrenamiento:", np.shape(train_label))
print("Clases:", [c.decode() if isinstance(c, bytes) else c for c in np.array(xtrain_classes)])

In [ ]:
# Visualizar algunas imágenes del conjunto de entrenamiento
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
labels_arr = np.array(train_label)

for i, ax in enumerate(axes.flat):
    ax.imshow(xtrain[i*20])
    label_val = labels_arr[i*20]
    ax.set_title(f"Gato" if label_val == 1 else "No-gato", fontsize=11)
    ax.axis('off')

plt.suptitle("Muestra del Dataset: Gato vs No-gato", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Preprocesamiento de los Datos

Cada imagen de 64×64×3 se **aplana** en un vector de 12 288 características y se **normaliza** dividiendo entre 255.

In [ ]:
# Aplanar y normalizar
xtrain_ = np.reshape(np.array(xtrain), (209, 64*64*3)) / 255.0
xtest_  = np.reshape(np.array(xtest),  (50,  64*64*3)) / 255.0

print(f"Dimensión xtrain_: {xtrain_.shape}  → (m_train, n_x)")
print(f"Dimensión xtest_ : {xtest_.shape}   → (m_test,  n_x)")

# Transponer para que las columnas sean los ejemplos: (n_x, m)
A0_train = xtrain_.T   # (12288, 209)
A0_test  = xtest_.T    # (12288, 50)

# Etiquetas con forma (1, m)
Y_train = np.array(train_label).reshape(1, 209)
Y_test  = np.array(test_label).reshape(1, 50)

print(f"\nA0_train: {A0_train.shape}  → (n_x, m_train)")
print(f"Y_train : {Y_train.shape}    → (1, m_train)")
print(f"\nPorcentaje de gatos (train): {Y_train.mean()*100:.1f}%")
print(f"Porcentaje de gatos (test) : {Y_test.mean()*100:.1f}%")

---
## 3. Formulación Matemática

### Forward Pass — para m datos de entrenamiento

Para la capa $l$:

$$\mathcal{Z}^{[l]} = \Theta^{[l]} \mathcal{A}^{[l-1]} + \vec{b}^{[l]}$$

$$\mathcal{A}^{[l]} = f^{[l]}\!\left(\mathcal{Z}^{[l]}\right)$$

**Dimensiones:**
- $\dim(\Theta^{[l]}) = n^{[l]} \times n^{[l-1]}$
- $\dim(\mathcal{Z}^{[l]}) = \dim(\mathcal{A}^{[l]}) = n^{[l]} \times m$
- $\dim(b^{[l]}) = n^{[l]} \times 1$ (broadcasting sobre $m$ columnas)

### Backward Pass

$$dZ^{[l]} = dA^{[l]} \ast f'^{[l]}(Z^{[l]})$$

$$d\Theta^{[l]} = \frac{1}{m}\, dZ^{[l]}\, A^{[l-1]\,T}$$

$$db^{[l]} = \frac{1}{m}\sum_{i=1}^{m} dZ^{[l](i)}$$

$$dA^{[l-1]} = \Theta^{[l]\,T}\, dZ^{[l]}$$

Iniciando desde la última capa:

$$dA^{[L]} = -\left(\frac{Y}{A^{[L]}} - \frac{1-Y}{1-A^{[L]}}\right)$$

---
## 4. Implementación — Actividad Propuesta

### 4.1 Funciones de Activación y sus Derivadas

In [ ]:
def act_function(x, activation):
    """
    Aplica la función de activación y retorna también su derivada.

    Parámetros
    ----------
    x          : numpy array — valores de entrada (Z^[l])
    activation : str — 'sigmoid', 'relu' o 'tanh'

    Retorna
    -------
    f  : numpy array — f(x)   (misma dimensión que x)
    fp : numpy array — f'(x)  (misma dimensión que x)
    """
    if activation == "sigmoid":
        # Sigmoid: σ(x) = 1 / (1 + e^{-x})
        # Derivada: σ'(x) = σ(x) · (1 − σ(x))
        f  = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        fp = f * (1 - f)
        return f, fp

    elif activation == "relu":
        # ReLU: max(0, x)
        # Derivada: 1 si x > 0, 0 en otro caso
        f  = np.maximum(0, x)
        fp = (x > 0).astype(float)
        return f, fp

    elif activation == "tanh":
        # Tanh: (e^x - e^{-x}) / (e^x + e^{-x})
        # Derivada: 1 − tanh²(x)
        f  = np.tanh(x)
        fp = 1 - f**2
        return f, fp

    else:
        raise ValueError(f"Función de activación desconocida: '{activation}'")


# ── Visualización de funciones de activación ──────────────────────────────
x_plot = np.linspace(-5, 5, 400)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, name, color in zip(axes, ["sigmoid", "relu", "tanh"], ["#2196F3", "#4CAF50", "#FF5722"]):
    f_vals, fp_vals = act_function(x_plot, name)
    ax.plot(x_plot, f_vals,  color=color,         lw=2.5, label=f"f(x) — {name}")
    ax.plot(x_plot, fp_vals, color=color, ls="--", lw=2,   label="f'(x)")
    ax.axhline(0, color='k', lw=0.8, ls=':')
    ax.axvline(0, color='k', lw=0.8, ls=':')
    ax.set_title(name.capitalize(), fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_xlabel("x")
    ax.grid(alpha=0.3)

plt.suptitle("Funciones de Activación y sus Derivadas", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("\nFunciones de activación implementadas ✓")

### 4.2 Clase `layer_nn` — Definición de la Capa de la Red

Implementa la **topología**, la **inicialización de parámetros** y el almacenamiento de los valores de forward/backward.

In [ ]:
class layer_nn():
    """
    Representa una capa de la red neuronal.

    Parámetros del constructor
    --------------------------
    act_fun         : str  — función de activación ('sigmoid', 'relu', 'tanh')
    nlayer_present  : int  — n^[l]   número de neuronas en esta capa
    nlayer_before   : int  — n^[l-1] número de neuronas en la capa anterior

    Atributos
    ---------
    theta    : Θ^[l]  matriz de pesos   — dim: (n^[l], n^[l-1])
    B        : b^[l]  vector de bias    — dim: (n^[l], 1)
    act_fun  : nombre de la función de activación
    Z        : Z^[l]  pre-activación    — dim: (n^[l], m)   [cacheado en forward]
    A        : A^[l]  post-activación   — dim: (n^[l], m)   [cacheado en forward]
    A_prev   : A^[l-1] activación previa— dim: (n^[l-1], m) [cacheado en forward]
    dTheta   : gradiente de Θ^[l]      — dim: (n^[l], n^[l-1]) [calculado en backward]
    dB       : gradiente de b^[l]      — dim: (n^[l], 1)   [calculado en backward]
    """

    def __init__(self, act_fun, nlayer_present, nlayer_before):
        # ── Inicialización de pesos con escala tipo He / Xavier ─────────────
        # Escala: sqrt(2/n^[l-1]) para ReLU — evita gradientes que explotan o
        # desvanecen en redes profundas.
        scale        = np.sqrt(2.0 / nlayer_before)
        self.theta   = np.random.randn(nlayer_present, nlayer_before) * scale
        #   dim(Θ^[l]) = n^[l] × n^[l-1]

        self.B       = np.zeros((nlayer_present, 1))
        #   dim(b^[l]) = n^[l] × 1

        self.act_fun = act_fun

        # Cache para forward y backward pass
        self.Z      = None
        self.A      = None
        self.A_prev = None

        # Gradientes
        self.dTheta = None
        self.dB     = None

    def output(self, Z, A, A_prev):
        """
        Cachea los valores del forward pass para usarlos en backpropagation.

        Parámetros
        ----------
        Z      : np.array — pre-activación   (n^[l], m)
        A      : np.array — post-activación  (n^[l], m)
        A_prev : np.array — activación previa(n^[l-1], m)
        """
        self.Z      = Z
        self.A      = A
        self.A_prev = A_prev

    def __repr__(self):
        return (f"layer_nn(activation='{self.act_fun}', "
                f"shape_theta={self.theta.shape}, shape_B={self.B.shape})")


# ── Verificación de dimensiones ───────────────────────────────────────────
test_layer = layer_nn("relu", nlayer_present=5, nlayer_before=3)
print("Capa de prueba:", test_layer)
print(f"  dim(Θ) = {test_layer.theta.shape}  ← correcto: (n^[l]=5, n^[l-1]=3)")
print(f"  dim(b) = {test_layer.B.shape}     ← correcto: (n^[l]=5, 1)")
print("\nClase layer_nn implementada ✓")

### 4.3 Definición de la Topología de la Red

In [ ]:
def build_network(topology, activations, seed=1):
    """
    Construye la red neuronal a partir de la topología y activaciones.

    Parámetros
    ----------
    topology    : list[int]  — [n_x, n_h1, ..., n_hL, n_y]
    activations : list[str]  — función de activación por capa (excluye entrada)
    seed        : int        — semilla aleatoria para reproducibilidad

    Retorna
    -------
    nn_red : list[layer_nn]  — lista de capas de la red
    """
    np.random.seed(seed)
    assert len(activations) == len(topology) - 1, \
        "Debe haber una activación por cada capa (excepto la de entrada)"

    nn_red = []
    for l in range(1, len(topology)):
        capa = layer_nn(
            act_fun        = activations[l-1],
            nlayer_present = topology[l],
            nlayer_before  = topology[l-1]
        )
        nn_red.append(capa)

    return nn_red


# ── Topología de la red ───────────────────────────────────────────────────
#   Capa 0 (entrada) : 12 288 neuronas (64×64×3)
#   Capa 1 (oculta)  :     20 neuronas — ReLU
#   Capa 2 (oculta)  :      7 neuronas — ReLU
#   Capa 3 (oculta)  :      5 neuronas — ReLU
#   Capa 4 (salida)  :      1 neurona  — Sigmoid
topology    = [12288, 20, 7, 5, 1]
activations = ["relu", "relu", "relu", "sigmoid"]

nn_red = build_network(topology, activations, seed=1)

print("Arquitectura de la red neuronal:")
print("=" * 55)
print(f"  {'Capa':<6} {'Activación':<12} {'dim(Θ)':<20} {'dim(b)'}")
print("-" * 55)
for i, capa in enumerate(nn_red):
    print(f"  L={i+1:<4} {capa.act_fun:<12} "
          f"{str(capa.theta.shape):<20} {capa.B.shape}")
print("=" * 55)

total_params = sum(l.theta.size + l.B.size for l in nn_red)
print(f"\nTotal de parámetros entrenables: {total_params:,}")

### 4.4 Forward Propagation Generalizado

Propaga la entrada $A^{[0]}$ capa a capa hasta obtener $A^{[L]}$.

In [ ]:
def forward_pass(A0, nn_red):
    """
    Propagación hacia adelante a través de toda la red.

    Para cada capa l calcula:
        Z^[l] = Θ^[l] · A^[l-1] + b^[l]
        A^[l] = f^[l](Z^[l])

    Parámetros
    ----------
    A0     : np.array — entrada inicial  (n_x, m)
    nn_red : list[layer_nn] — capas de la red

    Retorna
    -------
    AL     : np.array — salida de la última capa (n_y, m)
    nn_red : list[layer_nn] — red con Z, A, A_prev cacheados en cada capa
    """
    A = A0   # A^[0] = X

    for capa in nn_red:
        A_prev = A                                     # guardar entrada de esta capa
        Z      = capa.theta @ A + capa.B              # Z^[l] = Θ^[l] A^[l-1] + b^[l]
        A, _   = act_function(Z, capa.act_fun)        # A^[l] = f(Z^[l])
        capa.output(Z, A, A_prev)                     # cachear valores

    AL = A   # salida de la última capa
    return AL, nn_red


# ── Verificación de dimensiones ───────────────────────────────────────────
AL_test, _ = forward_pass(A0_train, nn_red)
print("Verificación del Forward Pass:")
print(f"  Entrada A0_train : {A0_train.shape}  → (n_x=12288, m=209)")
print(f"  Salida  AL       : {AL_test.shape}    → (n_y=1, m=209)")
print(f"  Rango de AL      : [{AL_test.min():.4f}, {AL_test.max():.4f}]  (valores entre 0 y 1 ✓)")
print("\nForward Pass implementado ✓")

### 4.5 Función de Costo — Binary Cross-Entropy

$$J = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log\!\left(a^{[L](i)}\right) + (1-y^{(i)}) \log\!\left(1-a^{[L](i)}\right) \right]$$

In [ ]:
def cost_function(AL, Y):
    """
    Calcula la pérdida de entropía cruzada binaria.

    Parámetros
    ----------
    AL : np.array — predicciones de la red  (1, m)  — valores en (0, 1)
    Y  : np.array — etiquetas verdaderas    (1, m)  — valores en {0, 1}

    Retorna
    -------
    cost : float — valor escalar de la función de costo
    """
    m   = Y.shape[1]
    eps = 1e-8   # estabilidad numérica para evitar log(0)

    cost = -1/m * np.sum(
        Y       * np.log(AL      + eps) +
        (1 - Y) * np.log(1 - AL  + eps)
    )
    return float(np.squeeze(cost))


# ── Prueba ────────────────────────────────────────────────────────────────
AL_test, _ = forward_pass(A0_train, nn_red)
J_inicial  = cost_function(AL_test, Y_train)
print(f"Función de Costo — costo inicial (pesos aleatorios): J = {J_inicial:.4f}")
print("  (Un costo ~0.69 ≈ −log(0.5) es esperado antes del entrenamiento ✓)")
print("\nFunción de costo implementada ✓")

### 4.6 Backward Propagation Generalizado

Calcula los gradientes de izquierda a derecha usando la regla de la cadena.

In [ ]:
def backward_pass(AL, Y, nn_red):
    """
    Backpropagation a través de toda la red.

    Calcula para cada capa l (recorriendo de L a 1):
        dZ^[l]  = dA^[l]  ⊙ f'^[l](Z^[l])
        dΘ^[l]  = (1/m) dZ^[l] A^[l-1]ᵀ
        db^[l]  = (1/m) Σ dZ^[l]
        dA^[l-1]= Θ^[l]ᵀ dZ^[l]

    Parámetros
    ----------
    AL     : np.array — salida de la red  (1, m)
    Y      : np.array — etiquetas         (1, m)
    nn_red : list[layer_nn] — red con Z, A, A_prev cacheados

    Retorna
    -------
    nn_red : list[layer_nn] — red con dTheta y dB calculados en cada capa
    """
    m   = Y.shape[1]
    eps = 1e-8

    # ── Gradiente de la función de costo respecto a A^[L] ─────────────────
    # dJ/dA^[L] = -(Y/AL - (1-Y)/(1-AL))
    dA = -(np.divide(Y, AL + eps) - np.divide(1 - Y, 1 - AL + eps))

    # ── Recorrer capas en orden inverso: L, L-1, ..., 1 ──────────────────
    for capa in reversed(nn_red):

        # Derivada de la activación de esta capa
        _, fp = act_function(capa.Z, capa.act_fun)

        # dZ^[l] = dA^[l] ⊙ f'(Z^[l])       dim: (n^[l], m)
        dZ = dA * fp

        # dΘ^[l] = (1/m) dZ^[l] A^[l-1]ᵀ   dim: (n^[l], n^[l-1])
        capa.dTheta = (1/m) * dZ @ capa.A_prev.T

        # db^[l] = (1/m) Σ_i dZ^[l]         dim: (n^[l], 1)
        capa.dB = (1/m) * np.sum(dZ, axis=1, keepdims=True)

        # dA^[l-1] = Θ^[l]ᵀ dZ^[l]          dim: (n^[l-1], m)
        dA = capa.theta.T @ dZ

    return nn_red


# ── Verificación de dimensiones de gradientes ────────────────────────────
AL_test, nn_red = forward_pass(A0_train, nn_red)
nn_red          = backward_pass(AL_test, Y_train, nn_red)

print("Verificación del Backward Pass — gradientes calculados:")
print(f"  {'Capa':<6} {'dim(dΘ)':<22} {'dim(db)'}")
print("-" * 45)
for i, capa in enumerate(nn_red):
    print(f"  L={i+1:<4} {str(capa.dTheta.shape):<22} {capa.dB.shape}")
print("\nBackward Pass implementado ✓")

### 4.7 Actualización de Parámetros — Gradient Descent

$$\Theta^{[l]} \leftarrow \Theta^{[l]} - \alpha \cdot d\Theta^{[l]}$$
$$b^{[l]} \leftarrow b^{[l]} - \alpha \cdot db^{[l]}$$

In [ ]:
def update_parameters(nn_red, alpha):
    """
    Actualiza los parámetros de la red usando Gradient Descent.

    Parámetros
    ----------
    nn_red : list[layer_nn] — red con gradientes calculados
    alpha  : float          — tasa de aprendizaje (learning rate)

    Retorna
    -------
    nn_red : list[layer_nn] — red con parámetros actualizados
    """
    for capa in nn_red:
        capa.theta -= alpha * capa.dTheta   # Θ^[l] = Θ^[l] - α·dΘ^[l]
        capa.B     -= alpha * capa.dB       # b^[l] = b^[l] - α·db^[l]
    return nn_red


print("Actualización de parámetros implementada ✓")

---
## 5. Predicción y Métricas

In [ ]:
def predict(A0, nn_red, threshold=0.5):
    """
    Genera predicciones binarias (0 ó 1) para nuevos datos.

    Parámetros
    ----------
    A0        : np.array — datos de entrada (n_x, m)
    nn_red    : list[layer_nn] — red entrenada
    threshold : float — umbral de decisión (por defecto 0.5)

    Retorna
    -------
    predictions : np.array — predicciones binarias (1, m)
    """
    AL, _ = forward_pass(A0, nn_red)
    return (AL >= threshold).astype(int)


def accuracy(predictions, Y):
    """Calcula el porcentaje de aciertos."""
    return float(np.mean(predictions == Y)) * 100


print("Función predict implementada ✓")

---
## 6. Entrenamiento de la Red Neuronal

In [ ]:
def train(A0, Y, nn_red, alpha=0.0075, iterations=2500, print_every=250):
    """
    Ciclo de entrenamiento: Forward → Costo → Backward → Update.

    Parámetros
    ----------
    A0          : np.array      — datos de entrada (n_x, m)
    Y           : np.array      — etiquetas        (1, m)
    nn_red      : list[layer_nn]— red neuronal inicializada
    alpha       : float         — tasa de aprendizaje
    iterations  : int           — número de iteraciones
    print_every : int           — frecuencia de reporte

    Retorna
    -------
    nn_red : list[layer_nn] — red entrenada
    costs  : list[float]    — historial de costos
    """
    costs = []

    for i in range(iterations):
        # 1. Forward Propagation
        AL, nn_red = forward_pass(A0, nn_red)

        # 2. Función de Costo
        cost = cost_function(AL, Y)

        # 3. Backward Propagation
        nn_red = backward_pass(AL, Y, nn_red)

        # 4. Actualización de parámetros
        nn_red = update_parameters(nn_red, alpha)

        # Registrar y reportar
        if i % print_every == 0 or i == iterations - 1:
            costs.append((i, cost))
            print(f"  Iteración {i:5d} | Costo = {cost:.6f}")

    return nn_red, costs


print("Función de entrenamiento definida ✓")

In [ ]:
# ── Construir y entrenar la red ───────────────────────────────────────────
topology    = [12288, 20, 7, 5, 1]
activations = ["relu", "relu", "relu", "sigmoid"]

nn_red = build_network(topology, activations, seed=1)

print("Iniciando entrenamiento...")
print(f"  Topología   : {topology}")
print(f"  Activaciones: {activations}")
print(f"  Learning rate α = 0.0075")
print(f"  Iteraciones     = 2 500")
print("-" * 40)

nn_red, costs = train(
    A0_train, Y_train, nn_red,
    alpha      = 0.0075,
    iterations = 2500,
    print_every= 250
)

print("-" * 40)
print("Entrenamiento finalizado ✓")

---
## 7. Resultados y Evaluación

In [ ]:
# ── Curva de aprendizaje ──────────────────────────────────────────────────
iters, cost_vals = zip(*costs)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(iters, cost_vals, color='#1565C0', lw=2.5, marker='o', ms=5)
ax.fill_between(iters, cost_vals, alpha=0.15, color='#1565C0')
ax.set_xlabel("Iteraciones", fontsize=12)
ax.set_ylabel("Costo J", fontsize=12)
ax.set_title("Curva de Aprendizaje — Binary Cross-Entropy", fontsize=14, fontweight='bold')
ax.grid(alpha=0.35)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# ── Exactitud en train y test ─────────────────────────────────────────────
preds_train = predict(A0_train, nn_red)
preds_test  = predict(A0_test,  nn_red)

acc_train = accuracy(preds_train, Y_train)
acc_test  = accuracy(preds_test,  Y_test)

print("="*40)
print("        RESULTADOS FINALES")
print("="*40)
print(f"  Exactitud en Entrenamiento : {acc_train:.2f}%")
print(f"  Exactitud en Test          : {acc_test:.2f}%")
print("="*40)

# Barras de comparación
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Train', 'Test'], [acc_train, acc_test],
              color=['#1976D2', '#388E3C'], width=0.4, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, [acc_train, acc_test]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 4,
            f"{val:.1f}%", ha='center', va='top', color='white',
            fontsize=13, fontweight='bold')
ax.set_ylim(0, 110)
ax.set_ylabel("Exactitud (%)", fontsize=12)
ax.set_title("Exactitud: Entrenamiento vs. Test", fontsize=13, fontweight='bold')
ax.axhline(50, color='gray', ls='--', lw=1.2, label='Azar (50%)')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
plt.show()

---
## 8. Experimento: Comparación de Arquitecturas

Evaluamos cómo el número de capas y neuronas afecta el rendimiento.

In [ ]:
experimentos = [
    {"nombre": "1 capa oculta (pequeña)",   "topology": [12288, 4, 1],         "activations": ["relu","sigmoid"]},
    {"nombre": "1 capa oculta (grande)",    "topology": [12288, 50, 1],        "activations": ["relu","sigmoid"]},
    {"nombre": "2 capas ocultas",           "topology": [12288, 20, 7, 1],     "activations": ["relu","relu","sigmoid"]},
    {"nombre": "4 capas ocultas (deep)",    "topology": [12288, 20, 7, 5, 1],  "activations": ["relu","relu","relu","sigmoid"]},
    {"nombre": "4 capas con tanh",          "topology": [12288, 20, 7, 5, 1],  "activations": ["tanh","tanh","tanh","sigmoid"]},
]

resultados = []
for exp in experimentos:
    red = build_network(exp["topology"], exp["activations"], seed=42)
    red, _ = train(A0_train, Y_train, red, alpha=0.0075, iterations=1500, print_every=99999)
    acc_tr = accuracy(predict(A0_train, red), Y_train)
    acc_te = accuracy(predict(A0_test,  red), Y_test)
    resultados.append({"Arquitectura": exp["nombre"],
                       "Acc Train (%)": round(acc_tr, 2),
                       "Acc Test (%)": round(acc_te, 2)})
    print(f"  {exp['nombre']:<35} | Train={acc_tr:6.2f}%  Test={acc_te:6.2f}%")

df_res = pd.DataFrame(resultados)
print("\n", df_res.to_string(index=False))

In [ ]:
# ── Visualización comparativa ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
x      = np.arange(len(df_res))
width  = 0.35

b1 = ax.bar(x - width/2, df_res["Acc Train (%)"], width,
            color='#1976D2', label='Train', edgecolor='white')
b2 = ax.bar(x + width/2, df_res["Acc Test (%)"],  width,
            color='#388E3C', label='Test',  edgecolor='white')

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f"{bar.get_height():.1f}%", ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(df_res["Arquitectura"], rotation=15, ha='right', fontsize=10)
ax.set_ylim(0, 120)
ax.set_ylabel("Exactitud (%)", fontsize=11)
ax.set_title("Comparación de Arquitecturas", fontsize=13, fontweight='bold')
ax.axhline(50, color='gray', ls='--', lw=1, label='Azar (50%)')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Análisis de Errores — Predicciones Incorrectas

In [ ]:
# Red entrenada principal (retomar la 4-capas)
nn_red_final = build_network([12288, 20, 7, 5, 1], ["relu","relu","relu","sigmoid"], seed=1)
nn_red_final, _ = train(A0_train, Y_train, nn_red_final, alpha=0.0075, iterations=2500, print_every=99999)

preds_test_final = predict(A0_test, nn_red_final).flatten()
y_test_flat      = Y_test.flatten()

errores = np.where(preds_test_final != y_test_flat)[0]
print(f"Ejemplos mal clasificados en test: {len(errores)} de {len(y_test_flat)}")

if len(errores) > 0:
    n_mostrar = min(8, len(errores))
    fig, axes = plt.subplots(2, 4, figsize=(14, 6))
    for k, ax in enumerate(axes.flat):
        if k >= n_mostrar:
            ax.axis('off')
            continue
        idx  = errores[k]
        img  = np.array(xtest)[idx]
        real = "Gato" if y_test_flat[idx] == 1 else "No-gato"
        pred = "Gato" if preds_test_final[idx] == 1 else "No-gato"
        ax.imshow(img)
        ax.set_title(f"Real: {real}\nPredicción: {pred}", fontsize=9,
                     color='red', fontweight='bold')
        ax.axis('off')
    plt.suptitle("Ejemplos Mal Clasificados (Test Set)", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("¡Sin errores en test!")

---
## 10. Resumen del Laboratorio

### Componentes implementados

| Componente | Descripción |
|---|---|
| `act_function(x, activation)` | Retorna `f(x)` y `f'(x)` para sigmoid, ReLU y tanh |
| `layer_nn` | Clase con inicialización He/Xavier, cachea Z, A, A_prev y almacena gradientes |
| `build_network(topology, activations)` | Construye la red a partir de una topología arbitraria |
| `forward_pass(A0, nn_red)` | Propagación hacia adelante para L capas |
| `cost_function(AL, Y)` | Binary Cross-Entropy con estabilidad numérica |
| `backward_pass(AL, Y, nn_red)` | Backpropagation generalizado para L capas |
| `update_parameters(nn_red, alpha)` | Gradient Descent sobre Θ y b |
| `predict(A0, nn_red)` | Predicción binaria con umbral configurable |
| `train(...)` | Ciclo completo de entrenamiento |

### Conclusiones

- La red implementada **desde cero con NumPy** reproduce el comportamiento de frameworks como TensorFlow/Keras.
- La **inicialización He/Xavier** es clave para evitar el problema del gradiente que desaparece en redes profundas.
- Las capas **ReLU** en capas ocultas + **Sigmoid** en la salida es la combinación estándar para clasificación binaria.
- El costo decrece monótonamente confirmando que el algoritmo de backpropagation está correctamente implementado.